# TD11: Linguistic laws

## 0. Question

Today, we will answer the following questions:

- Does the Brown corpus follow the rank-frequency pattern predicted by Zipf's law?
- Are frequent words in the Brown corpus shorter on average than less frequent words (Zipf's law of abbreviation)?

We will use the Brown corpus from NLTK, a classic balanced corpus of written English, and we will work through the analysis step by step.

In [ ]:
import nltk # corpus access
import numpy as np # numerical computation
import pandas as pd # data-frame manipulation
import matplotlib.pyplot as plt # plotting
import seaborn as sns # plotting
from nltk.corpus import brown # Brown corpus
from scipy.optimize import curve_fit # fitting the curve
from sklearn.metrics import r2_score # coefficient of determination

sns.set(context='paper', style='ticks',
        font_scale=1, palette='colorblind')

If the Brown corpus is not installed in your environment yet, run the following cell:

In [ ]:
try:
    brown.ensure_loaded()
except LookupError:
    nltk.download('brown')
    brown.ensure_loaded()

## 1. Data

### 1.1. Loading the Brown corpus

Let's start by loading all word tokens from the Brown corpus.

In [ ]:
tokens = brown.words()
len(tokens)

What do the first tokens look like?

In [ ]:
tokens[:40]

### 1.2. Cleaning the corpus

For this practical, we will only keep alphabetic tokens and we will lowercase everything. This removes punctuation and makes frequencies easier to interpret. 

Convert tokens to lowercase and keep only alphabetic tokens.

In [ ]:
##################
# YOUR CODE HERE #
##################

Let's inspect the cleaned tokens.

In [ ]:
words[:40]

How many cleaned word tokens and unique words do we have?

In [ ]:
##################
# YOUR CODE HERE #
##################

### 1.3. Building a frequency table

Now we will count how often each word type occurs in the corpus.

In [ ]:
freq_table = (
    pd.Series(words, name='word')
      .value_counts()
      .rename_axis('word')
      .reset_index(name='frequency')
)

freq_table.head(10)

Let's add the rank of each word and its length in letters. Name the columns `rank` and `length` respectively. How can you add the rank? Can you use the `index` of the DataFrame?

In [ ]:
##################
# YOUR CODE HERE #
##################

## 2. Zipf's law

### 2.1. A first look at the rank-frequency distribution

We first plot the rank and frequency of a 100 most frequent words on the original scale:

In [ ]:
##################
# YOUR CODE HERE #
##################

The decay is easier to see on log-log axes. Modify your initial plot to use log-log axes. Use the plt.xscale and plt.yscale functions to set the axes to logarithmic scale.

In [ ]:
##################
# YOUR CODE HERE #
##################

### 2.2. Checking rank x frequency

A simple rule-of-thumb version of Zipf's law says that `rank x frequency` should be roughly stable for a large part of the distribution.

In [ ]:
zipf_check = freq_table.head(1000).copy()
zipf_check['rank_x_frequency'] = zipf_check['rank'] * zipf_check['frequency']

zipf_check[['rank', 'frequency', 'rank_x_frequency']].head(10)

Let's summarize this quantity.

In [ ]:
zipf_check['rank_x_frequency'].describe()

And now let's plot it. You need to plot rank times frequency against rank. What do you see? Is it stable? 

In [ ]:
##################
# YOUR CODE HERE #
##################

### 2.3. Fitting Zipf's law

To keep the fit readable, we will focus on the 2000 most frequent words. We first fit the basic power-law model

$$f(r) = A / r^b$$

In [ ]:
fit_data = freq_table.head(2000).copy()

ranks = fit_data['rank'].to_numpy(dtype=float)
frequencies = fit_data['frequency'].to_numpy(dtype=float)

Frequencies are stored as counts, let's convert them to relative frequencies by dividing by the total number of tokens. Store them in the `relative_freqquencies` variable.

In [ ]:
##################
# YOUR CODE HERE #
##################

Now let's define the original Zipf's law:

In [ ]:
def zipf_basic(r, A, b):
    return A / (r ** b)

Let's fit the original Zipf's law:

In [ ]:
params, _ = curve_fit(
    zipf_basic,
    ranks,
    relative_frequencies,
    p0=(frequencies[0], 1.0),
    bounds=(0, np.inf),
    maxfev=10000
)

A_basic, b_basic = params
basic_prediction = zipf_basic(ranks, A_basic, b_basic)

# keep these for the next cell
log_ranks = np.log(ranks)
log_frequencies = np.log(frequencies)

print(f"Basic model: f(r) = {A_basic:.1f} / r^{b_basic:.2f}")

Let's now plot the preidctions:

In [ ]:
plt.figure(figsize=(8, 4))

# fitted curve
plt.plot(ranks, basic_prediction, 
         label="Zipf's law", color='orange',
         linewidth=5, alpha=0.5)

# observed points as scatter
plt.scatter(ranks, relative_frequencies, 
            alpha=0.4, s=18, label='Brown corpus',
            color='blue')

# apply symlog scales after plotting
plt.xscale('log')
plt.yscale('log')

plt.xlabel('Rank')
plt.ylabel('Relative frequency')
sns.despine()
plt.legend()
plt.show()

Now let's compute the coefficient of determination $R^2$ for the original Zipf's law:

In [ ]:
r2_basic = r2_score(relative_frequencies, basic_prediction)
print(f"R² (basic Zipf model): {r2_basic:.4f}")

Now let's try the Zipf-Mandelbrot law, which adds two parameters to the original Zipf's law:

$$f(r) = A / (r + c)^b$$

Repeat the fitting and plotting steps for the Zipf-Mandelbrot law, based on what we did above. 

In [ ]:
##################
# YOUR CODE HERE #
##################

Now let's plot the Zipf-Mandelbrot predictions:

In [ ]:
##################
# YOUR CODE HERE #
##################

Let's compute the $R^2$ for the Zipf-Mandelbrot law:

In [ ]:
##################
# YOUR CODE HERE #
##################

Which model fits better? Why do you think that is?

## 3. Zipf's law of abbreviation

### 3.1. Preparing the data

For this part, we will use the same frequency table, but we will focus on words that occur at least 5 times. This makes the pattern less noisy.

In [ ]:
abbreviation_data = freq_table.query('frequency >= 5').copy()
abbreviation_data.shape

Let's inspect the beginning of this table.

In [ ]:
abbreviation_data[['word', 'frequency', 'length']].head(10)

### 3.2. Visual inspection

We can now plot word length against frequency. Since frequency is very skewed, we will use a logarithmic x-axis.

In [ ]:
##################
# YOUR CODE HERE #
##################

### 3.3. A permutation test

A simple way to test the Zipf's law of abbreviation is to compare the observed value of

$$\sum_i \text{length}_i \times \text{frequency}_i$$

against a null distribution where word lengths are randomly reassigned to frequencies.

In [ ]:
lengths = abbreviation_data['length'].to_numpy(dtype=float)
frequencies = abbreviation_data['frequency'].to_numpy(dtype=float)

observed = np.sum(lengths * frequencies)
observed

Let's build the null distribution by shuffling the lengths 2000 times. You would need to build an array of 2000 values of the above sum, where in each iteration you shuffle the lengths and compute the sum again. Use np.random.permutation to shuffle the lengths.

In [ ]:
np.random.seed(42)
null_statistics = np.empty(2000)

for i in range(len(null_statistics)):
    ##################
    # YOUR CODE HERE #
    ##################
    pass

What is the mean and standard deviation of the null distribution? Is it bigger or smaller than the observed value? What does it mean?

In [93]:
##################
# YOUR CODE HERE #
##################

Finally, let's visualize the null distribution against the observed value.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(null_statistics, bins=30, alpha=0.75, color='steelblue',
         label='simulated')
plt.axvline(observed, color='crimson', linewidth=2, label='observed')
plt.xlabel('Sum(length x frequency)')
plt.ylabel('Count')
plt.title('Permutation test for the Brown corpus')
plt.legend()
plt.show()